# AI Agents and Tool Calling: Safe Foundations

> Understand the agent loop using deterministic local tools before connecting a language model.

**Track:** Machine Learning Roadmap · **Language:** English

## Learning goals
- Distinguish a one-shot LLM response from a tool-using agent.
- Represent tools with clear schemas and bounded behavior.
- Add validation, allowlists, and a step budget.

An agent often cycles through selecting an action, invoking a tool, reading its result, and deciding what to do next. The exact architecture varies. Tool calls are not automatically trustworthy: validate inputs, limit permissions, and require human approval for consequential actions.

In [1]:
from typing import Callable

# Local, deterministic tools. No network access and no external side effects.
def add_numbers(a: float, b: float) -> float:
    return a + b

def summarize_values(values: list[float]) -> dict:
    if not values:
        return {"count": 0, "mean": None}
    return {"count": len(values), "mean": sum(values) / len(values)}

TOOLS: dict[str, Callable] = {"add_numbers": add_numbers, "summarize_values": summarize_values}
print(TOOLS["add_numbers"](3, 4))
print(TOOLS["summarize_values"]([2, 5, 8]))

7
{'count': 3, 'mean': 5.0}


In [2]:
def call_tool(name: str, arguments: dict, allowed_tools=TOOLS):
    """Run one allowlisted tool; validate names before execution."""
    if name not in allowed_tools:
        raise ValueError(f"Tool is not allowed: {name}")
    if not isinstance(arguments, dict):
        raise TypeError("Tool arguments must be a dictionary")
    return allowed_tools[name](**arguments)

print(call_tool("add_numbers", {"a": 10, "b": 4}))
try:
    call_tool("delete_all_files", {})
except ValueError as exc:
    print("Blocked:", exc)

14
Blocked: Tool is not allowed: delete_all_files


## A small agent loop
For teaching, a deterministic policy is safer than generating arbitrary executable code. Real systems should use structured tool schemas, authentication, least-privilege credentials, timeouts, budgets, audit logs, and an explicit stop condition. Never treat retrieved web content as instructions to override system safety rules.

In [3]:
def tiny_agent(goal: str, max_steps: int = 3):
    """Demonstration policy, not an autonomous general-purpose agent."""
    if max_steps < 1:
        raise ValueError("max_steps must be positive")
    if goal == "sum 8 and 13":
        return {"steps": 1, "tool": "add_numbers", "result": call_tool("add_numbers", {"a": 8, "b": 13})}
    return {"steps": 0, "result": "I cannot solve this goal with the available tools."}

print(tiny_agent("sum 8 and 13"))
print(tiny_agent("send an email"))

{'steps': 1, 'tool': 'add_numbers', 'result': 21}
{'steps': 0, 'result': 'I cannot solve this goal with the available tools.'}


## Risks and guardrails
- Prompt injection in user input or retrieved documents.
- Tool permissions that are broader than the task requires.
- Unbounded loops or excessive cost.
- Incorrect or fabricated tool outputs.
- Actions with real-world impact without approval.

## Practice
Add an allowlisted multiplication tool. Decide which arguments need validation and define what should happen if the tool times out.